# ASK Audio Demo on PYNQ-Z2

This notebook loads `ask_audio.bit`, initializes the onboard ADAU1761 codec, loads baseband symbols into IFM BRAM, and starts the ASK modulator. Probe the PYNQ-Z2 headphone output with an oscilloscope.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if (repo_root / 'pynq').exists():
    sys.path.insert(0, str(repo_root / 'pynq'))
else:
    sys.path.insert(0, str(Path.cwd()))

from ask_audio_demo import AskAudioDemo, pattern_symbols, print_addressable_ips, plot_debug_capture, save_debug_capture_csv, summarize_debug_capture
from pynq import Overlay

BITFILE = 'ask_audio.bit'
# Slow hardware debug mode: chosen so AXI polling can draw a readable waveform.
# For target audio mode later, use CARRIER_HZ=4000 and SYMBOL_RATE=100.
CARRIER_HZ = 5
SYMBOL_RATE = 1
SYMBOL_COUNT = 32
PATTERN = [0, 1, 3, 2]


In [ ]:
# Diagnostic: verify the .hwh exposes both ask_audio_top_0 and axi_bram_ctrl_0.
ol_check = Overlay(BITFILE)
print_addressable_ips(ol_check)


In [ ]:
symbols = pattern_symbols(SYMBOL_COUNT, PATTERN)
demo = AskAudioDemo(BITFILE, init_codec=True)
demo.stop()
load_verify = demo.load_symbols(symbols, verify=True)
config = demo.configure(carrier_hz=CARRIER_HZ, symbol_rate=SYMBOL_RATE, symbol_count=len(symbols))
demo.start(loop=True)
{'config': config, 'bram_load_verify': load_verify, 'first_16_symbols': demo.read_symbols(16)}


In [ ]:
demo.status(), demo.read_debug()


In [ ]:
# First RTL debug plot from AXI status registers.
# This is a slow software snapshot, not a cycle-accurate waveform capture.
debug_samples = demo.capture_debug_samples(count=1000, interval_s=0.005)
save_debug_capture_csv(debug_samples, 'ask_debug_capture.csv')
print(summarize_debug_capture(debug_samples))
fig, axes = plot_debug_capture(debug_samples)
fig


Oscilloscope starting point:

- Probe the headphone output, AC-coupled first.
- Time base: 1 ms/div to 5 ms/div.
- Expected carrier: 4 kHz.
- Expected envelope change: 100 symbols/s using the repeating 00, 01, 11, 10 pattern.
- If output is silent, check `demo.read_debug()` and use ILA on `ask_out`, `symbol_valid_dbg`, `codec_lrclk`, and `codec_sdata_o`.